In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Pass Manager로 트랜스파일하기

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
아래 버전 이상을 사용하는 것을 권장합니다.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</details>
Circuit을 트랜스파일하는 권장 방법은 staged pass manager를 생성한 후, Circuit을 입력으로 하여 `run` 메서드를 실행하는 것입니다. 이 페이지에서는 이 방법으로 양자 Circuit을 트랜스파일하는 방법을 설명합니다.
## (Staged) Pass Manager란 무엇인가요?
Qiskit SDK의 맥락에서 트랜스파일이란, 입력 Circuit을 양자 디바이스에서 실행 가능한 형태로 변환하는 과정을 말합니다. 트랜스파일은 일반적으로 트랜스파일러 pass라고 불리는 일련의 단계로 이루어집니다. Circuit은 각 트랜스파일러 pass를 순서대로 처리하며, 한 pass의 출력이 다음 pass의 입력이 됩니다. 예를 들어, 한 pass는 Circuit을 순회하며 연속된 단일 Qubit Gate 시퀀스를 병합하고, 다음 pass는 이 Gate들을 대상 디바이스의 기저 집합으로 합성할 수 있습니다. Qiskit에 포함된 트랜스파일러 pass는 [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) 모듈에 위치합니다.

Pass manager는 트랜스파일러 pass 목록을 저장하고 Circuit에서 실행할 수 있는 객체입니다. 트랜스파일러 pass 목록으로 [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager)를 초기화하여 pass manager를 생성합니다. Circuit에서 트랜스파일을 실행하려면, Circuit을 입력으로 하여 [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) 메서드를 호출합니다.

Staged pass manager는 일반 pass manager보다 한 단계 높은 추상화 수준을 나타내는 특별한 종류의 pass manager입니다. 일반 pass manager가 여러 트랜스파일러 pass로 구성되는 반면, staged pass manager는 여러 *pass manager*로 구성됩니다. 트랜스파일은 [Transpiler 단계](transpiler-stages)에서 설명한 것처럼 일반적으로 개별 단계로 이루어지며, 각 단계는 pass manager로 표현되기 때문에 이러한 추상화는 유용합니다. Staged pass manager는 [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) 클래스로 표현됩니다. 이 페이지의 나머지 부분에서는 (staged) pass manager를 생성하고 사용자 지정하는 방법을 설명합니다.
## 사전 설정된 Staged Pass Manager 생성하기
합리적인 기본값으로 사전 설정된 staged pass manager를 생성하려면, [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 함수를 사용합니다:

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Circuit 또는 Circuit 목록을 pass manager로 트랜스파일하려면, Circuit 또는 Circuit 목록을 `run` 메서드에 전달합니다. Hadamard Gate 뒤에 인접한 두 개의 CX Gate로 구성된 2-Qubit Circuit으로 실행해 봅시다:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

`generate_preset_pass_manager` 함수에 가능한 인수에 대한 설명은 [트랜스파일 기본값 및 구성 옵션](defaults-and-configuration-options)을 참조하세요. `generate_preset_pass_manager`의 인수는 [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile) 함수의 인수와 동일합니다.

<CodeAssistantAdmonition tagLine="Pass manager 세부 사항이 기억나지 않으시나요? Qiskit Code Assistant에 물어보세요." />

사전 설정된 pass manager가 필요에 맞지 않는 경우, (staged) pass manager 또는 트랜스파일 pass를 직접 생성하여 트랜스파일을 사용자 지정할 수 있습니다. 이 페이지의 나머지 부분에서는 pass manager를 생성하는 방법을 설명합니다. 트랜스파일 pass를 생성하는 방법은 [나만의 Transpiler pass 작성하기](custom-transpiler-pass)를 참조하세요.

## 나만의 Pass Manager 생성하기

[qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) 모듈에는 pass manager를 생성하는 데 사용할 수 있는 많은 트랜스파일러 pass가 포함되어 있습니다. pass manager를 생성하려면, pass 목록으로 `PassManager`를 초기화합니다. 예를 들어, 다음 코드는 인접한 2-Qubit Gate를 병합한 후 [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate), [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate) 기저로 합성하는 트랜스파일러 pass를 생성합니다.

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

이 pass manager를 실제로 시연하기 위해, Hadamard Gate 뒤에 인접한 두 개의 CX Gate로 구성된 2-Qubit Circuit에서 테스트해 봅니다:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Circuit에서 pass manager를 실행하려면, `run` 메서드를 호출합니다.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

dynamical decoupling로 알려진 오류 억제 기법을 구현하기 위한 pass manager를 생성하는 방법을 보여주는 고급 예시는 [dynamical decoupling를 위한 pass manager 생성하기](dynamical-decoupling-pass-manager)를 참조하세요.

## Staged Pass Manager 생성하기

`StagedPassManager`는 개별 단계로 구성된 pass manager이며, 각 단계는 `PassManager` 인스턴스로 정의됩니다. 원하는 단계를 지정하여 `StagedPassManager`를 생성할 수 있습니다. 예를 들어, 다음 코드는 `init`과 `translation`의 두 단계로 구성된 staged pass manager를 생성합니다. `translation` 단계는 이전에 생성한 pass manager로 정의됩니다.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

staged pass manager에 넣을 수 있는 단계의 수에는 제한이 없습니다.

staged pass manager를 생성하는 또 다른 유용한 방법은 사전 설정된 staged pass manager로 시작한 후 일부 단계를 교체하는 것입니다. 예를 들어, 다음 코드는 최적화 레벨 3으로 사전 설정된 pass manager를 생성한 후, 사용자 정의 `pre_layout` 단계를 지정합니다.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[stage generator 함수](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions)는 사용자 정의 pass manager를 구성하는 데 유용할 수 있습니다.
이 함수들은 많은 pass manager에서 공통적으로 사용되는 기능을 제공하는 단계를 생성합니다.
예를 들어, [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager)는 레이아웃 pass에서 선택된 초기 `Layout`을 지정된 대상 디바이스에 "임베드"하는 단계를 생성하는 데 사용할 수 있습니다.

## 다음 단계
> **Tip:** - [사용자 정의 Transpiler pass 작성하기](custom-transpiler-pass).
>     - [dynamical decoupling를 위한 pass manager 생성하기](dynamical-decoupling-pass-manager).
>     - `generate_preset_passmanager` 함수에 대해 더 알아보려면, [트랜스파일 기본 설정 및 구성 옵션](defaults-and-configuration-options) 항목을 참조하세요.
>     - [Transpiler 설정 비교](/guides/circuit-transpilation-settings) 가이드를 시도해 보세요.
>     - [Transpiler API 문서](https://docs.quantum.ibm.com/api/qiskit/transpiler)를 검토하세요.